# Linux Shell and Bash Fundamentals (Educational Notebook)
This notebook introduces the shell: what it is, how to type and run commands correctly, and how environment variables shape the environment every command runs in.

## 1. What Is a Shell?

A **shell** is a program that reads commands you type (or reads them from a script file) and asks the kernel to carry them out - starting programs, managing files, connecting programs together. It sits between the user and the kernel, acting as a command interpreter.

There have been many Unix shells over the years (`sh`, `csh`, `ksh`, `zsh`, `fish`, ...). **Bash** ("Bourne Again SHell") is a GNU Project reimplementation of the original Unix Bourne shell (`sh`) and is the default interactive shell on most Linux distributions.

```bash
echo $SHELL          # the shell configured as your login shell
cat /etc/shells         # the list of shells installed and considered valid login shells
chsh -s /bin/zsh           # change your default login shell (takes effect on next login)
```


## 2. The Bash Prompt and Basic Interaction

When you open a terminal, Bash prints a **prompt** and waits for input. A typical prompt looks like:

```
user@hostname:~$
```

- `user` - the logged-in username.
- `hostname` - the machine's name.
- `~` - the current working directory (`~` is shorthand for the home directory).
- `$` - indicates a regular user; a `#` prompt instead usually indicates you are root.

Bash reads one line at a time, splits it into words, and treats the first word as the command to run and the rest as its arguments. Pressing `Tab` triggers **completion** (finishing a partially-typed command, filename, or option), and the Up/Down arrow keys move through previously typed commands.


## 3. Command Syntax: Commands, Options, and Arguments

The general shape of a shell command is:

```
command [options] [arguments]
```

- The **command** is the program (or shell builtin) to run.
- **Options** (also called flags or switches) modify its behavior, usually starting with `-` (short form, can often be combined: `-la`) or `--` (long form: `--all`).
- **Arguments** are the things the command should act on, such as filenames.

```bash
ls -l /etc                # command: ls, option: -l, argument: /etc
ls -la                      # combined short options: -l and -a together
grep -i "error" log.txt       # option -i (case-insensitive), argument log.txt
cp --recursive src/ dst/        # long-form option
```

Almost every command supports `--help`, and most have a manual page:

```bash
ls --help
man ls
```


## 4. Command Types: Builtins, Aliases, Functions, and External Programs

Not everything you type at the shell is a separate program on disk. Bash resolves each command word into one of several categories, in a fixed priority order:

1. **Aliases** - user-defined shorthand for another command (e.g. `alias ll='ls -la'`).
2. **Shell functions** - reusable blocks of shell code defined with `name() { ...; }`.
3. **Shell builtins** - commands implemented directly inside Bash itself (`cd`, `echo`, `export`, `pwd`, `type`, ...) rather than as separate executables. Builtins run faster (no new process) and some, like `cd`, *must* be builtins, since a separate process could never change the shell's own working directory.
4. **External programs** - executables found by searching the directories listed in `$PATH` (e.g. `/usr/bin/grep`).

```bash
type ls          # reports how a name resolves: alias, builtin, function, or a file path
type cd
which grep         # shows the path to the external program that would run (not aware of aliases/builtins)
```


## 5. Aliases

An **alias** lets you define a short name for a longer command. Aliases are convenient for interactive use but are not inherited by scripts unless explicitly enabled, so scripts should not rely on them.

```bash
alias ll='ls -la'          # define an alias for the current session
alias grep='grep --color=auto'
unalias ll                   # remove an alias
alias                          # list all currently defined aliases
```

To make an alias persist across sessions, add the `alias` line to your shell's startup file (`~/.bashrc`), covered in the next lesson.


## 6. Environment Variables and the Shell Environment

A **variable** is a named piece of data the shell can store and reuse. An **environment variable** is a variable that has been *exported*, meaning it is passed automatically into the environment of every child process the shell starts (not just used inside the shell itself).

```bash
NAME="value"          # set a plain shell variable (local to this shell)
echo $NAME               # read a variable's value (note the $ prefix when reading)
export NAME               # promote it to an environment variable, inherited by child processes
env                          # list all environment variables
printenv NAME                  # print the value of one specific environment variable
unset NAME                       # remove a variable entirely
```

Some important, commonly-used environment variables:

| Variable | Meaning |
|---|---|
| `HOME` | Path to the current user's home directory |
| `USER` | Current username |
| `PATH` | Directories searched, in order, to find external commands |
| `PWD` | Current working directory |
| `SHELL` | Path to the user's default login shell |
| `HOSTNAME` | The machine's hostname |


## 7. `PATH` and Command Lookup

`PATH` is a colon-separated list of directories that Bash searches, in order, when you type the name of an external command without giving its full path.

```bash
echo $PATH
# example output: /usr/local/bin:/usr/bin:/bin:/usr/local/sbin:/usr/sbin:/sbin

export PATH="$HOME/bin:$PATH"     # prepend a personal bin directory, searched first
```

If a command "isn't found" even though the program exists, it usually means either the program's directory isn't in `PATH`, or the program isn't executable (see the permissions lessons in `week3/`). Running a program by its full or relative path (`./script.sh`, `/opt/tool/run`) always works regardless of `PATH`, since no search is needed.


## 8. Setting and Exporting Variables

There's an important distinction between a **shell variable** and an **environment variable**:

```bash
GREETING="hello"        # shell variable: visible in this shell only
echo $GREETING             # works: hello
bash -c 'echo $GREETING'     # empty: the child shell never saw it, it wasn't exported

export GREETING="hello"        # environment variable: exported to children
bash -c 'echo $GREETING'         # works: hello, inherited by the child process
```

Rule of thumb: use a plain variable for something local to the current script or session; `export` a variable when a program you're about to launch (including another shell, or any external command) needs to see it.


## Hands-on

Try these on your own system:

```bash
echo $SHELL
type cd
type ls
which ls
alias ll='ls -la'
ll
FRUIT="apple"
echo $FRUIT
bash -c 'echo $FRUIT'      # notice this prints nothing
export FRUIT
bash -c 'echo $FRUIT'      # now it prints apple
echo $PATH
```

Predict the output of each command before running it, then compare.


## Review Questions

1. What does a shell actually do, conceptually, between you pressing Enter and a program running?
2. Why must `cd` be implemented as a shell builtin rather than as a separate external program?
3. In `grep -i "error" log.txt`, identify the command, the option, and the argument.
4. When you type a command name, in what order does Bash check aliases, functions, builtins, and external programs?
5. What is the practical difference between a plain shell variable and an exported environment variable?
6. If typing a command gives "command not found" even though the program is installed, what are two possible explanations?
7. Why shouldn't a shell script rely on the user's aliases being defined?
8. What does `which` show you, and why might it disagree with what actually runs if a name is also defined as an alias or function?


# Cheat Sheet

```
Identify things:
  echo $SHELL         current login shell
  type <name>           how a name resolves (alias/function/builtin/file)
  which <name>            path to the external program (ignores aliases/builtins)

Command shape:
  command [options] [arguments]      -l / -la   short options
                                       --all      long option

Aliases:
  alias ll='ls -la'    unalias ll    alias   (list all)

Variables:
  NAME=value            plain shell variable (local)
  export NAME=value       environment variable (inherited by children)
  echo $NAME                read a value
  env / printenv NAME         list / print environment variables
  unset NAME                    remove a variable

PATH:
  echo $PATH                       colon-separated search list
  export PATH="$HOME/bin:$PATH"      prepend a directory
```
